<a href="https://colab.research.google.com/github/EssaHusary/CSC803_Capstone_Project/blob/main/Capstone_Project_Backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount project to Google Drive and create "capstone_project" directory

In [ ]:
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd '/content/drive/MyDrive'

In [ ]:
%mkdir capstone_project

In [ ]:
# Change the working directory to somewhere in your Google Drive.
# You could check the path by right clicking on the folder.
%cd capstone_project/

In [ ]:
%pip install --upgrade websockets

In [ ]:
!pip show websockets

# Download the LLMs from HuggingFace

In [ ]:
file_path1 = "/content/drive/MyDrive/capstone_project/Qwen2.5-7B-Instruct-Q4_K_M.gguf"
if os.path.exists(file_path1):
  print("Good to go")
else:
  !wget -N https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf


In [ ]:
file_path2 = "/content/drive/MyDrive/capstone_project/Meta-Llama-3-8B-Instruct.Q4_K_M.gguf"
if os.path.exists(file_path2):
  print("Good to go")
else:
  !wget -N https://huggingface.co/QuantFactory/Meta-Llama-3-8B-Instruct-GGUF/resolve/main/Meta-Llama-3-8B-Instruct.Q4_K_M.gguf


In [ ]:
# These download the private and public data sets used to perform our experiment. For the purpose of the demo and chatbot, this cell can be ignored entirely.

# from pathlib import Path

# if not Path('./public.txt').exists():
#     !wget https://raw.githubusercontent.com/nipun-taneja/theory-of-computing-gen-ai/refs/heads/main/public.txt
# if not Path('./private.txt').exists():
#     !wget https://raw.githubusercontent.com/nipun-taneja/theory-of-computing-gen-ai/refs/heads/main/private.txt

In [ ]:
# This enables you to run LLMs locally.
!python3 -m pip install --no-cache-dir llama-cpp-python==0.3.4 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122


In [ ]:
# To allow automatic Google searches.
!python3 -m pip install googlesearch-python bs4 charset-normalizer requests-html lxml_html_clean

In [ ]:
# To check if a GPU is being used.
import torch
if not torch.cuda.is_available():
    raise Exception('You are not using the GPU runtime. Change it first or you will suffer from the super slow inference speed!')
else:
    print('You are good to go!')

# Load the models onto the GPU

In [ ]:
from llama_cpp import Llama
import os

# Load the model onto GPU
try:
    qwen2_5 = Llama(
        "./Qwen2.5-7B-Instruct-Q4_K_M.gguf",
        verbose=False,
        n_gpu_layers=-1, # Loads the entire model onto the GPU.
        n_ctx=16384      # This argument is how many tokens the model can take.
    )
except ValueError as e:
    print(f"Error loading Qwen model: {e}")
    model_path = "./Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    if not os.path.exists(model_path):
        print(f"The model file was not found at: {model_path}. Please ensure it exists and the path is correct.")

In [ ]:
from llama_cpp import Llama
import os

# Load the model onto GPU
try:
    llama3 = Llama(
        "./Meta-Llama-3-8B-Instruct.Q4_K_M.gguf",
        verbose=False,
        n_gpu_layers=-1, # Loads the entire model onto the GPU.
        n_ctx=16384      # This argument is how many tokens the model can take.
    )
except ValueError as e:
    print(f"Error loading Llama model: {e}")
    model_path = "./Meta-Llama-3-8B-Instruct.Q4_K_M.gguf"
    if not os.path.exists(model_path):
        print(f"The model file was not found at: {model_path}. Please ensure it exists and the path is correct.")

# Define the functions necessary for inferencing.

In [ ]:
def generate_response(_model: Llama, _messages: str) -> str:
    '''
    This function will inference the model with given messages.
    '''
    _output = _model.create_chat_completion(
        _messages,
        stop=["<|eot_id|>", "<|end_of_text|>"],
        max_tokens=512,     # This argument is how many tokens the model can generate.
        temperature=0,      # This argument is the randomness of the model. 0 means no randomness.
        repeat_penalty=2.0,
    )["choices"][0]["message"]["content"]
    return _output

In [ ]:
# Imports for search tool
!pip install ddgs

In [ ]:
# Search Tool Code
from ddgs import DDGS

async def ddgs_search(query: str, region= "us-en", max_results: int = 5):
  try:
    with DDGS() as ddgs:
      results = ddgs.text(query, region = region, max_results=max_results)
      reponse_text = ''
      for res in results :
        reponse_text += res['body'] + '\n'

      return reponse_text
  except Exception as e:
      print(f"An error occurred during DDGS search: {e}")
      return []

# Sample Run
print(await ddgs_search("Who was the first emperor of China?"))

In [ ]:
class LLMAgent():
    def __init__(self, role_description: str, task_description: str, llm:str="llama3"):
        self.role_description = role_description   # Role means who this agent should act like.
        self.task_description = task_description    # Task description instructs what task should this agent solve.
        self.llm = llm  # LLM indicates which LLM backend this agent is using.

    def inference(self, message:str) -> str:
        messages = [
            {"role": "system", "content": f"{self.role_description}"},
            {"role": "user", "content": f"{self.task_description}\n{message}"},
        ]

        if self.llm == 'llama3':
            return generate_response(llama3, messages)
        elif self.llm == 'qwen2_5':
            return generate_response(qwen2_5, messages)
        else:
            return ""

# Design the role description and task description for each agent.

### Agents using Llama

In [ ]:
# The first two agents in this cell are used for the strong pipeline found in the next couple of cells. Because the strong pipeline is used only in the experiment, these
# first two agents can be ignored. The third agent, "qa_agent", is used in the simple pipeline, so it is not commented out.

# # This agent may help you filter out the irrelevant parts in question descriptions.
# question_extraction_agent = LLMAgent(
#     role_description="You are an AI that simplifies a question from a long question description.",
#     task_description="Given a question, reformulate it. This reformulated question must retain some context of the original question. The reformulated question MUST NOT request additional detail(s) that are not present in the original question.",
#     llm='llama3'
# )

# # This agent may help you extract the keywords in a question so that the search tool can find more accurate results.
# keyword_extraction_agent = LLMAgent(
#     role_description="You are an AI that extracts key words from a question.",
#     task_description="Given a question, extract key words from it. These key words must be CRUCIAL in answering the question when seaching the internet. Extract as many key words as possible. GIVE ME ONLY THE KEY WORDS. DO NOT GIVE ME YOUR THINKING.",
#     llm='llama3'
# )

# This agent is the core component that answers the question.
qa_agent = LLMAgent(
    role_description="You are LLaMA-3.1-8B, an AI designed to answer questions. When using English, you will only answer questions in English.",
    task_description="Please answer the following questions:",
    llm='llama3'
)

### Agents using Qwen

In [ ]:
# This agent judges the output of another agent.
judgment_agent = LLMAgent(
    role_description="You are an AI agent that judges the output of another agent. The output is an answer to a question.",
    task_description="Please judge the output of the other agent. If the output is inaccurate, provide the correct answer. Please make sure that your spelling and grammar are correct.",
    llm='qwen2_5'
)

# This agent may help you extract the keywords in a question so that the search tool can find more accurate results.
keyword_extraction_agent_qwen = LLMAgent(
    role_description="You are an AI that extracts key words from a question.",
    task_description="Given a question, extract key words from it. These key words must be CRUCIAL in answering the question when seaching the internet. Extract as many key words as possible. GIVE ME ONLY THE KEY WORDS. DO NOT GIVE ME YOUR THINKING.",
    llm='qwen2_5'
)

### The pipelines involved in the verification pipeline.

In [ ]:
# Llama will provide its answer to a user's question.

def pipeline(question: str) -> str:
    # question: The question the user wants an answer to.

    questionConstraints = f"Answer this question: '{question}', using BETWEEN 10 and 20 words MAXIMUM."


    return qa_agent.inference(questionConstraints)

In [ ]:
# This function is used for the strong pipeline, which is used in the experiment, and can thus be ignored entirely.

# async def strong_pipeline(question: str) -> str:
#     # question: The question the user wants an answer to.

#     # To extract keywords from the question
#     keywords = keyword_extraction_agent.inference(question)

#     # To await the results from the web search
#     response = await ddgs_search(keywords)

#     # To refine the question for the LLM
#     refinedQuestion = question_extraction_agent.inference(question)

#     strongQuery = f"The following information was obtained from the internet: {response}. Using this information IN CONJUNCTION WITH your own knowledge, answer this question, '{question}', and it's reformulated version, '{refinedQuestion}'. Don't give me your reasoning, don't say, 'The answers to both questions is...', don't mention that a reformulated question exists, and don't say that you couldn't find a piece of information. DO NOT MENTION ANY SEARCH RESULTS and DO NOT MENTION 'PROVIDED INFORMATION'! Just give me the answer, using BETWEEN 10 words MINIMUM and 20 words MAXIMUM."
#     return qa_agent.inference(strongQuery)

In [ ]:
# This is how the verification pipeline works. A user question is fed into the simple pipeline. Qwen uses its own agents to extract key words to perform an internet search - the RAG process.
# Afterwards, the judgement agent (Qwen) is prompted to perform two judgements. The first judgement judges Llama's output via Qwen's own internal knowledge. The second judgement compares Llama's
# output with the retrieved search results. The corrected answer, if Llama's output needs correcting, is fed back to the user.

async def verification_pipeline(question):
  # question: The question the user wants an answer to.

  # Llama's output to the question
  answer = pipeline(question)

  # To extract keywords from the question to perform RAG
  keywords = keyword_extraction_agent_qwen.inference(question)

  # To await the results from the web search
  response = await ddgs_search(keywords)

  # For experiment. You can ignore this.
  # correctedAnswer = f"A large language model was asked this question: '{question}'. This was its answer: '{answer}'. Please judge the accuracy of its answer in two ways and reply to me with the correct answer: 1) In the first judgment, judge the accuracy of this answer by using your own internal knowledge. 2) In the second judgement, judge its answer by comparing its answer with these retreived search results: '{response}'. Using these two judgements, determine if the answer the large language model produced is accurate. If the answer is accurate, do nothing to the answer; simply tell me its answer and ONLY its answer. DO NOT tell me whether it is accurate or not. Now, if the answer is inaccurate, DO NOT tell me that is inaccurate or incorrect; simply correct its answer and simply tell me what the answer is. No matter what your judgement is, DO NOT, and I mean DO NOT, comment on the other large language models answer no matter if it's accurate or inaccurate! DO NOT MENTION ANY SEARCH RESULTS! Just TELL me the answer, using BETWEEN 10 words MINIMUM and 20 words MAXIMUM!"


  # For user
  correctedAnswer = f"A large language model was asked this question: '{question}'. This was its answer: '{answer}'. Please judge the accuracy of its answer in two ways and reply to me with the correct answer: 1) In the first judgment, judge the accuracy of this answer by using your own internal knowledge. 2) In the second judgement, judge its answer by comparing its answer with these retreived search results: '{response}'. Using these two judgements, determine if the answer the large language model produced is accurate. If the answer is accurate, do nothing to the answer; simply tell me its answer and ONLY its answer. DO NOT tell me whether it is accurate or not. Now, if the answer is inaccurate, DO NOT tell me that is inaccurate or incorrect; simply correct its answer and simply tell me what the answer is. No matter what your judgement is, DO NOT, and I mean DO NOT, comment on the other large language models answer no matter if it's accurate or inaccurate! DO NOT MENTION ANY SEARCH RESULTS! Just TELL me the answer, using BETWEEN 10 words MINIMUM and 20 words MAXIMUM!"

  return judgment_agent.inference(correctedAnswer)

# Connect the backend and frontend

In [ ]:
# These install the packages to enable you to generate a URL to connect the backend and frontend
!pip install fastapi uvicorn nest-asyncio
!npm install -g localtunnel

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

app = FastAPI()  # To create the FastAPI application instance

# To enable CORS middleware for connection between backend and frontend
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000"  # Only enable communication between a frontend that uses this port
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class Message(BaseModel):
    text: str


# This is the endpoint that receives user queries
@app.post("/chat")
async def chat(message: Message):

    # This is where our verification pipeline takes user input and returns verified output
    reply = await verification_pipeline(message.text)
    print(reply, flush=True)
    return {
        "reply": reply
    }


# To begin running the server
def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


threading.Thread(
    target=run_server,
    daemon=True
).start()

In [ ]:
# Creates a new URL using the name "chatbot2". You can change this name to "chatbot" or "monkey", if you want
!lt --port 8000 --subdomain chatbot2

# Experiment data collection (can be ignored entirely for demo purposes)

In [ ]:
# import csv
# from pathlib import Path


# columns = ['Question', 'SimplePipelineAnswer', 'StrongPipelineAnswer', 'VerificationPipelineAnswer']

# with open('./Capstone_Experiment_Generated_Answers3.csv', mode='w') as output_csv:
#   writer = csv.writer(output_csv)
#   writer.writerow(columns)


# with open('./public.txt', 'r') as input_f:
#     questions = input_f.readlines()
#     questions = [l.strip().split(',')[0] for l in questions]
#     with open('./Capstone_Experiment_Generated_Answers3.csv', mode='a') as output_csv:
#       writer = csv.writer(output_csv)
#       for id, question in enumerate(questions, 1):

#           answer1 = pipeline(question)
#           answer1 = answer1.replace('\n',' ')
#           answer2 = await strong_pipeline(question)
#           answer2 = answer2.replace('\n',' ')
#           answer3 = await verification_pipeline(question)
#           answer3 = answer3.replace('\n',' ')
#           writer.writerow([question, answer1, answer2, answer3])

#           print(id, answer1)
#           print(id, answer2)
#           print(id, answer3)

# with open('./private.txt', 'r') as input_f:
#     questions = input_f.readlines()
#     with open('./Capstone_Experiment_Generated_Answers3.csv', mode='a') as output_csv:
#       writer = csv.writer(output_csv)
#       for id, question in enumerate(questions, 1):

#           answer1 = pipeline(question)
#           answer1 = answer1.replace('\n',' ')
#           answer2 = await strong_pipeline(question)
#           answer2 = answer2.replace('\n',' ')
#           answer3 = await verification_pipeline(question)
#           answer3 = answer3.replace('\n',' ')
#           writer.writerow([question, answer1, answer2, answer3])

#           print(id, answer1)
#           print(id, answer2)
#           print(id, answer3)